<a href="https://colab.research.google.com/github/tettty1981-lang/data-analytics-course/blob/main/HW_15_3_%D0%A2%D0%B5%D1%81%D1%82%D0%B8_%D0%B4%D0%BB%D1%8F_%D0%BC%D0%B0%D0%BB%D0%B8%D1%85_%D0%B2%D0%B8%D0%B1%D1%96%D1%80%D0%BE%D0%BA_%D1%82%D0%B0_%D0%BF%D1%80%D0%BE%D0%BF%D0%BE%D1%80%D1%86%D1%96%D0%B9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Тести для малих вибірок та пропорцій



In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats



**Завдання 1**. E-commerce компанія після редизайну сайту підозрює, що **середній час до покупки (time-to-purchase)** користувачів **збільшився**.
Історично середній час від першого заходу на сайт до покупки становив $\mu_0 = 123.7$ хвилин.

Команда аналітиків випадково обрала дані **7 покупців після редизайну**:
`128, 135, 121, 142, 126, 151, 123`

З допомогою наявних даних зʼясуйте, чи збільшився середній час покупки після редизайну.

Для цього виконайте наступні 6 кроків. Правильне виконання кожного з кроків оцінюється в 1 бал.

1. Запишіть нульову та альтернативну гіпотези і визначте тип тесту.
2. Обчисліть вибіркові статистики: $\bar x$, $s$, $n$.
3. Оберіть тип тесту та виконайте його (знайдіть критичне значення тесту, статистику тесту та р-значення) будь-яким пасуючим способом, щоб перевірити гіпотезу на рівні значущості $\alpha = 0.10$.
4. Прийміть рішення, чи відхиляєте ви гіпотезу $H_0$ використовуючи p-value.
5. Напишіть висновок: чи справді редизайн сайту подовжив час до покупки?
6. Чи зміниться ваше рішення при зміні рівня значущості на $\alpha = 0.05$.


**Гіпотези**

H₀: μ = 123.7 (до редизайну середній час від першого заходу на сайт до покупки)

H₁: μ > 123.7 (середній час після редизайну)
Тип тесту: правосторонній (one-tailed)

In [2]:
# Вихідні дані
data = [128, 135, 121, 142, 126, 151, 123]

In [3]:
# Дані з умови
mu_0 = 123.7   # μ₀ — історичний середній час до покупки (хв)
alpha = 0.10   # рівень значущості

In [8]:
# Вибіркові статистики: x̄, n
n = len(data)
x_bar = sum(data) / n

In [9]:
# Відхилення від x̄ і їхні квадрати
deviations = [x - x_bar  for x in data]
squares = [d ** 2 for d in deviations]


In [10]:
# Вибіркове стандартне відхилення s (ділимо на n - 1)
sample_var = sum(squares) / (n - 1)
sample_std = np.sqrt(sample_var)

In [11]:
print("n =", n)
print("x̄ =", x_bar)
print("Σ квадратів =", sum(squares))
print("s =", sample_std)

n = 7
x̄ = 132.28571428571428
Σ квадратів = 723.4285714285714
s = 10.980502200328935


In [14]:
# Обчислимо t-статистику вручну
t_stat = (x_bar - mu_0) / (sample_std / np.sqrt(n))


In [24]:
# Критичне значення для правостороннього тесту
df = n - 1
t_crit = stats.t.ppf(1 - alpha, df)


In [25]:
# Рахуємо p-value
p_value = 1 - stats.t.cdf(t_stat, df)

In [27]:
print(f"Вибіркове середнє: {x_bar:.2f}")
print(f"Вибіркове стандартне відхилення: {sample_std:.2f}")
print(f"t-статистика: {t_stat:.3f}")
print(f"Критичне значення (df={df}): {t_crit:.3f}")
print(f"p-value: {p_value:.4f}")

# Висновок
if p_value < alpha:
    print("Відхиляємо H0")
else:
    print("Не відхиляємо H0")

Вибіркове середнє: 132.29
Вибіркове стандартне відхилення: 10.98
t-статистика: 2.069
Критичне значення (df=6): 1.440
p-value: 0.0420
Відхиляємо H0


**Висновок:** на рівні значущості 0.10 є достатньо статистичних доказів, що середній час до покупки після редизайну збільшився порівняно з історичним значенням 123.7 хв.

Для α = 0.05 висновок той самий.

**Завдання 2.**

До спеціальної рекламної кампанії **23%** дорослих упізнавали логотип компанії. Після завершення кампанії відділ маркетингу провів опитування: з **1200** випадково відібраних дорослих **311** упізнали логотип.

Перевірте на рівні значущості **3%** ($\alpha=0.03$), чи дають ці дані достатні підстави стверджувати, що **тепер більше ніж 23%** дорослих упізнають логотип компанії. Для розвʼязку використовуйте бібліотеку `statsmodels`.

Зробіть висновок, чи зросла впізнаваність логотипу.

Додатково, обчисліть довірчий інтревал на заданому рівні значущості і проінтерпретуйте текстом - як він додатково пояснює прийняте нами рішення?

In [36]:
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest, proportion_confint

count = 311   # кількість "успіхів"
nobs = 1200    # обсяг вибірки
p0 = 0.23      # гіпотетична пропорція (H0)
alpha = 0.03

stat, pval = proportions_ztest(count, nobs, p0, alternative='larger')
confint = proportion_confint(count, nobs, alpha=0.03, method='normal')

print(f"Z-статистика = {stat:.3f}")
print(f"p-value = {pval:.4f}")
print(f"{(1-alpha)*100:.0f}% довірчий інтервал: {confint}")


Z-статистика = 2.306
p-value = 0.0106
97% довірчий інтервал: (0.23171700302179205, 0.28661633031154127)


**Гіпотеза:**

H0: p = 0.23

H1: p > 0.23

In [35]:
if pval < alpha:
		print("Відхиляємо H0")
else:
		print("Немає підстав відхилити H0")

Відхиляємо H0


**Висновок:** на рівні значущості 3% є достатньо доказів, що впізнаваність логотипу зросла понад 23%, а 97% довірчий інтервал (0.2317; 0.2866) не містить 0.23, що підтверджує це рішення.